# SkinIQ — multi-label annotation

Label **all six skin concerns** for each photo: acne, blackheads, hyperpigmentation, redness,
enlarged pores, wrinkles. Clear skin = nothing ticked.

**Read the rules first:** [annotation guidelines](https://github.com/rwiie/SkinIQ/blob/main/docs/annotation_guidelines.md).

- Photos come in priority order: **300 test** photos, then **200 validation**, then training.
  About **1,200 photos** (≈2 hours) is a good minimum; you can stop and come back any time.
- Every click is saved straight to `My Drive/SkinIQ/labels/multilabel_labels.csv`.
  Re-run this notebook later and it continues where you stopped.
- No GPU needed: **Runtime → Change runtime type → CPU** is fine.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/SkinIQ"
SKIN_TYPE_ZIP = "Skin Type Dataset - olimov.zip"
SKIN_CONDITION_ZIP = "Skin Issues Dataset - drishya.zip"
ANNOTATOR = "raneem"          # your name, saved with each label

## 1. Set up (about 2 minutes)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
for z in (SKIN_TYPE_ZIP, SKIN_CONDITION_ZIP):
    assert os.path.exists(f"{DRIVE_DIR}/{z}"), f"Not found: {DRIVE_DIR}/{z}"

%cd /content
!rm -rf SkinIQ && git clone -q https://github.com/rwiie/SkinIQ.git
%cd /content/SkinIQ
!mkdir -p data/raw
!cp "{DRIVE_DIR}/{SKIN_TYPE_ZIP}" "{DRIVE_DIR}/{SKIN_CONDITION_ZIP}" /tmp/
!unzip -q -o "/tmp/{SKIN_TYPE_ZIP}" -x "__MACOSX/*" -d data/raw
!unzip -q -o "/tmp/{SKIN_CONDITION_ZIP}" -x "__MACOSX/*" -d data/raw
!rm -rf data/raw/skin_type data/raw/skin_condition
!mv "data/raw/Skin Type Dataset - olimov" data/raw/skin_type
!mv "data/raw/Skin Issues Dataset - drishya" data/raw/skin_condition
!python training/prepare_data.py

# Check the photos match the annotation queue exactly
import hashlib, pandas as pd
queue = pd.read_csv("data/labels/annotation_queue.csv")
bad = [p for p, m in zip(queue.image, queue.md5)
       if hashlib.md5(open(f"data/processed/{p}", "rb").read()).hexdigest() != m]
assert not bad, f"{len(bad)} photos differ from the queue — tell Claude"
print(f"Ready: {len(queue)} photos in the queue")

## 2. Label

Click the concerns you see (blue = ticked), then **Save & next ▶**. Use **◀ Back** to fix a mistake.

In [ ]:
import sys
sys.path.insert(0, "training")
from annotation import AnnotationSession, LabelStore, build_ui
from IPython.display import display

store = LabelStore(f"{DRIVE_DIR}/labels/multilabel_labels.csv")
session = AnnotationSession(queue, store, annotator=ANNOTATOR)
display(build_ui(session, "data/processed"))

## 3. Summary (run any time)

In [ ]:
labels = pd.read_csv(f"{DRIVE_DIR}/labels/multilabel_labels.csv")
usable = labels[labels.unusable == 0]
print(f"Labelled {len(labels)} photos ({len(labels) - len(usable)} unusable, {int(usable.unsure.sum())} unsure)")
concerns = ["acne", "blackheads", "hyperpigmentation", "redness", "enlarged_pores", "wrinkles"]
summary = usable.groupby("split")[concerns].sum()
summary["clear (none ticked)"] = usable.assign(clear=usable[concerns].sum(axis=1) == 0).groupby("split")["clear"].sum()
summary["photos"] = usable.groupby("split").size()
display(summary)